# Council of 8 — K-ZERO Console
### 8 minds. 1 question. Infinite consequences.

Drop Elon Musk, Richard Feynman, Kobe Bryant, Steve Jobs, Sartre, George Carlin, Bryan Johnson, and a Korean PhD student into a room. Ask a question. Watch them collide.

**What you need:** A free Groq API key from [console.groq.com](https://console.groq.com) (no credit card required).

In [ ]:
!pip install openai plotly dash rich python-dotenv numpy --quiet

# Clone the repo (uncomment if running on Kaggle)
# !git clone https://github.com/YOUR_USERNAME/the-council.git
# %cd the-council

import os, sys
sys.path.insert(0, '.')  # Add project root to path

In [ ]:
import os
# Get your free key at console.groq.com (no credit card)
os.environ["LLM_API_KEY"] = "YOUR_GROQ_KEY_HERE"  # <-- PASTE YOUR KEY
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["COUNCIL_MODEL"] = "llama-3.3-70b-versatile"
os.environ["COUNCIL_TEMPERATURE"] = "0.85"
os.environ["COUNCIL_MAX_TOKENS"] = "500"

In [ ]:
from runner.validate import validate
validate(".")

## Run the Deliberation
The council will debate: **"What is the point of life?"**
2 agents speak per round. God-mode injects stressors mid-discussion.

In [ ]:
from runner.council_runner import CouncilRunner

runner = CouncilRunner(".", scenario_path="scenarios/scenario_01_meaning_of_life.json")
runner.run(max_rounds=5, speakers_per_round=2)

In [ ]:
from runner.analyze import analyze_transcript
from pathlib import Path

# Find the latest transcript
latest = sorted(Path("transcripts").glob("*.json"))[-1]
print(f"Analyzing: {latest.name}")
result = analyze_transcript(str(latest))

# Show key findings
import json
print("\n=== KEY QUOTES ===")
for agent, q in result.get("key_quotes", {}).items():
    print(f"\n{agent}:")
    print(f'  "{q["quote"]}"')
    print(f"  Round {q['round']} — {q['why_it_matters']}")

print("\n=== EMERGENT INSIGHTS ===")
for insight in result.get("emergent_insights", []):
    print(f"\n• {insight['insight']}")
    print(f"  Contributors: {', '.join(insight['contributing_agents'])}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import math

# --- Extract data from analysis result ---
agent_names = result["agreement_matrix"]["labels"]
matrix = np.array(result["agreement_matrix"]["matrix"])
n = len(agent_names)

# Short labels for readability
short_names = [
    name.split("(")[0].strip().split()[-1] if "(" in name else name.split()[-1]
    for name in agent_names
]

# Agent colors matching the runner
agent_colors = {
    "Elon Musk": "#e74c3c",
    "Richard Feynman": "#3498db",
    "Kobe Bryant": "#f1c40f",
    "Steve Jobs": "#ecf0f1",
    "Jean-Paul Sartre": "#9b59b6",
    "George Carlin": "#2ecc71",
    "Bryan Johnson": "#2980b9",
    "Kevin (김경선)": "#bdc3c7",
}
node_colors = [agent_colors.get(name, "#95a5a6") for name in agent_names]

# =============================================
# FIGURE 1: Network Graph (Agreement/Conflict)
# =============================================

# Arrange nodes in a circle
angles = [2 * math.pi * i / n for i in range(n)]
node_x = [math.cos(a) for a in angles]
node_y = [math.sin(a) for a in angles]

# Build edges
edge_traces = []
for i in range(n):
    for j in range(i + 1, n):
        score = matrix[i][j]
        if abs(score) < 0.1:
            continue  # Skip near-zero edges
        # Color: green for agreement, red for conflict
        if score > 0:
            color = f"rgba(46, 204, 113, {min(abs(score), 1.0)})"
        else:
            color = f"rgba(231, 76, 60, {min(abs(score), 1.0)})"
        width = max(1, abs(score) * 5)
        edge_traces.append(
            go.Scatter(
                x=[node_x[i], node_x[j], None],
                y=[node_y[i], node_y[j], None],
                mode="lines",
                line=dict(width=width, color=color),
                hoverinfo="text",
                text=f"{short_names[i]} ↔ {short_names[j]}: {score:+.2f}",
                showlegend=False,
            )
        )

# Node trace
node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    marker=dict(size=30, color=node_colors, line=dict(width=2, color="#2c3e50")),
    text=short_names,
    textposition="top center",
    textfont=dict(size=11, color="white"),
    hoverinfo="text",
    hovertext=[f"{name}" for name in agent_names],
    showlegend=False,
)

fig_network = go.Figure(data=edge_traces + [node_trace])
fig_network.update_layout(
    title=dict(text="Council of 8 — Agreement Network", font=dict(size=18, color="white")),
    showlegend=False,
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#0f0f23",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-1.5, 1.5]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-1.5, 1.5]),
    width=700,
    height=600,
    margin=dict(l=40, r=40, t=60, b=40),
    annotations=[
        dict(text="Green = agreement, Red = conflict, Width = intensity",
             x=0.5, y=-0.05, xref="paper", yref="paper",
             showarrow=False, font=dict(size=10, color="#7f8c8d")),
    ],
)

fig_network.show()

# =============================================
# FIGURE 2: 8x8 Agreement Heatmap
# =============================================

fig_heatmap = go.Figure(
    data=go.Heatmap(
        z=matrix.tolist(),
        x=short_names,
        y=short_names,
        colorscale=[
            [0.0, "#c0392b"],   # strong disagreement
            [0.25, "#e74c3c"],
            [0.5, "#2c3e50"],    # neutral
            [0.75, "#27ae60"],
            [1.0, "#2ecc71"],    # strong agreement
        ],
        zmid=0,
        zmin=-1,
        zmax=1,
        text=[[f"{matrix[i][j]:+.2f}" for j in range(n)] for i in range(n)],
        texttemplate="%{text}",
        textfont=dict(size=10, color="white"),
        hovertemplate="%{y} ↔ %{x}: %{z:+.2f}<extra></extra>",
        colorbar=dict(
            title="Score",
            tickvals=[-1, -0.5, 0, 0.5, 1],
            ticktext=["Conflict", "", "Neutral", "", "Agreement"],
            tickfont=dict(color="white"),
            titlefont=dict(color="white"),
        ),
    )
)

fig_heatmap.update_layout(
    title=dict(text="Council of 8 — Pairwise Agreement Matrix", font=dict(size=18, color="white")),
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#0f0f23",
    xaxis=dict(tickfont=dict(color="white"), tickangle=45),
    yaxis=dict(tickfont=dict(color="white"), autorange="reversed"),
    width=700,
    height=600,
    margin=dict(l=80, r=40, t=60, b=80),
)

fig_heatmap.show()

print("\nDone. Two figures rendered inline.")